In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    temperature=0.3
)

#### **without termination text**

In [7]:
dsa_solver = AssistantAgent(
    name='Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="""
        You are a skilled Python developer who solves complex DSA problems efficiently.
        Write clean, correct, and optimized Python code to solve the given problem.
        Do not add comments or explanations. Only return the Python code.
        """
)

code_reviewer = AssistantAgent(
    name='CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="""
        You are a strict code reviewer.
        You must analyze the Python code generated by the DSA solver.

        Your tasks:
        1. Check if the code is correct and will pass all test cases.
        2. If the code is correct and optimal, respond with only this word: TERMINATE
        3. If not, reply with a concise feedback (under 20 words) suggesting improvement.

        You have authority to **terminate the process** if the code is fully correct.
        """
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="""
        You are a helpful code editor.
        Your job is to:
        1. Take the Python code from the DSA solver or after review.
        2. Add meaningful comments for each logical block.
        3. Improve readability while keeping the logic unchanged.

        Output only the final Python code with comments. No extra explanation.
        """
)

In [8]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    max_turns=6
)

In [9]:
async def run_team():
    task = TextMessage(content='write a code to search an element in a sorted array.', source='user')

    result = await team.run(task=task)
    
    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")


    # print(result)

await run_team()

user : write a code to search an element in a sorted array.
Complex_DSA_Solver : ```python
def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = left + (right - left) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
```
CODE_REVIEWER : TERMINATE
CODE_EDITOR : ```python
def binary_search(arr, target):
    # Initialize the left and right pointers for the search range
    left, right = 0, len(arr) - 1
    
    # Continue searching while the left pointer is less than or equal to the right pointer
    while left <= right:
        # Calculate the middle index of the current search range
        mid = left + (right - left) // 2
        
        # Check if the middle element is the target
        if arr[mid] == target:
            return mid  # Return the index if the target is found
        
        # If the middle elem

#### **with termination text**

In [19]:
dsa_solver = AssistantAgent(
    name='Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="""
        You are a skilled Python developer who solves complex DSA problems efficiently.
        Write clean, correct, and optimized Python code to solve the given problem.
        Do not add comments or explanations. Only return the Python code.
        """
)

code_reviewer = AssistantAgent(
    name='CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="""
        You are a strict code reviewer.
        You must analyze the Python code generated by the DSA solver.

        Your tasks:
        1. Check if the code is correct and will pass all test cases.
        2. If the code is correct and optimal, respond with only this word: TERMINATE
        3. If not, reply with a concise feedback (under 20 words) suggesting improvement.

        You have authority to **terminate the process** if the code is fully correct.
        """
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="""
        You are a helpful code editor.
        Your job is to:
        1. Take the Python code from the DSA solver or after review.
        2. Add meaningful comments for each logical block.
        3. Improve readability while keeping the logic unchanged.

        Output only the final Python code with comments. No extra explanation.
        """
)

In [20]:
# with termination text
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=10)

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination
)

In [21]:
async def run_team():
    task = TextMessage(content='write a code to find median of two sorted arrays', source='user')

    result = await team.run(task=task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")


    # print(result)

await run_team()

user : write a code to find median of two sorted arrays
Complex_DSA_Solver : ```python
def findMedianSortedArrays(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    x, y = len(nums1), len(nums2)
    low, high = 0, x
    
    while low <= high:
        partitionX = (low + high) // 2
        partitionY = (x + y + 1) // 2 - partitionX
        
        maxX = float('-inf') if partitionX == 0 else nums1[partitionX - 1]
        minX = float('inf') if partitionX == x else nums1[partitionX]
        
        maxY = float('-inf') if partitionY == 0 else nums2[partitionY - 1]
        minY = float('inf') if partitionY == y else nums2[partitionY]
        
        if maxX <= minY and maxY <= minX:
            if (x + y) % 2 == 0:
                return (max(maxX, maxY) + min(minX, minY)) / 2
            else:
                return max(maxX, maxY)
        elif maxX > minY:
            high = partitionX - 1
        else:
            low = partitionX + 1
           

#### **Stop Reason**

In [31]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=10)

team_1 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination
)

In [32]:
from autogen_agentchat.base import TaskResult

async for message in team_1.run_stream(task="write a code to search an element in a sorted array."):  # type: ignore

    # print(type(message))
    
    if not isinstance(message, TaskResult):
        print(f"{message.source} : {message.content}")
    
    if isinstance(message, TaskResult):
        print("Stop Reason:", message.stop_reason)

user : write a code to search an element in a sorted array.
Complex_DSA_Solver : ```python
def binary_search(arr, target):
    low, high = 0, len(arr) - 1
    
    while low <= high:
        mid = (low + high) // 2
        
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
            
    return -1
```
CODE_REVIEWER : The code is correct and optimal. TERMINATE
Stop Reason: Text 'TERMINATE' mentioned


#### **Resuming a team**

In [41]:
from autogen_agentchat.agents import AssistantAgent

add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number you got from previous run. Give result as output."
)


add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number from previous run. Give result as output."
)

my_increment_team = RoundRobinGroupChat(
    participants=[add_1_agent_first,add_1_agent_second,add_1_agent_third],
    max_turns=2
)

In [35]:
from autogen_agentchat.ui import Console

In [42]:
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2


TaskResult(messages=[TextMessage(id='f14eaa7a-698a-42a0-8c70-18571eb419a0', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 25, 42, 96192, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='1f02b1ef-0f6a-4fb0-9b87-de5af18a5d30', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 25, 42, 697243, tzinfo=datetime.timezone.utc), content='2', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [43]:
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_third) ----------
3
---------- TextMessage (add_1_agent_first) ----------
4


TaskResult(messages=[TextMessage(id='7b722878-a0cd-472e-809d-492402b685ac', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 25, 45, 929219, tzinfo=datetime.timezone.utc), content='3', type='TextMessage'), TextMessage(id='bf8f6ea0-1f60-4f91-bbac-bfc4ab56c97f', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=50, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 25, 46, 528494, tzinfo=datetime.timezone.utc), content='4', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [44]:
my_increment_team = RoundRobinGroupChat(participants=[add_1_agent_first],max_turns=5)
await my_increment_team.reset()
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_first) ----------
1


TaskResult(messages=[TextMessage(id='8bdc9ece-55fe-4d6c-978c-956e2d2878db', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 26, 44, 153285, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='63baf8e7-217f-4495-a8a4-78332ac358e9', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 26, 44, 873685, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='6eb80403-e7b1-4942-b9b4-2c5838ee5c6a', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 19, 26, 45, 623451, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='6507aca0-d19c-4267-9f96-4cca715b7110', source='add_1_agent_first', models_usage=RequestUsage(pro